In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Geometry-V7 R0/F1 thin handoff
Phase A is fail-closed until the reviewed runner commit is pushed and Phase B binds its approved GitHub exact. It never loads code from Drive and writes results to a new Drive directory only.

In [ ]:
import re
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
APPROVED_EXACT = '1eadbbb8d2dee0ab6cb6c85170b55966049af450'
if re.fullmatch(r'[0-9a-f]{40}', APPROVED_EXACT) is None:
    raise RuntimeError('Bind the approved pushed Geometry-V7 exact before execution')
checkout = Path('/content/CEG-WM')
if checkout.exists():
    raise FileExistsError(f'create-only checkout already exists: {checkout}')
subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', APPROVED_EXACT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(checkout)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torchmetrics', 'lpips'], check=True)

In [ ]:
import os
import torch
from google.colab import userdata

assert torch.cuda.is_available(), 'GPU required; no R0 result on CPU'
LOCAL_RESULT_DIR = Path('/content/geometry_v7_r0_result')
SYNCSEAL_CHECKPOINT = Path('/content/checkpoints/syncmodel.jit.pt')
DRIVE_RESULT_DIR = Path('/content/drive/MyDrive/CEG-WM/Geometry-V7') / APPROVED_EXACT / 'r0-f1'
if LOCAL_RESULT_DIR.exists() or SYNCSEAL_CHECKPOINT.exists():
    raise FileExistsError('create-only local R0 path already exists')
if DRIVE_RESULT_DIR.exists():
    raise FileExistsError(f'create-only Drive result already exists: {DRIVE_RESULT_DIR}')
root_key = userdata.get('CEG_WM_ROOT_KEY')
hf_token = userdata.get('HF_TOKEN')
if not all(isinstance(value, str) and value.strip() for value in (root_key, hf_token)):
    raise RuntimeError('CEG_WM_ROOT_KEY and HF_TOKEN are required')
markers = ('TOKEN', 'KEY', 'SECRET', 'PASSWORD', 'CREDENTIAL')
runner_env = {
    name: value for name, value in os.environ.items()
    if not any(marker in name.upper() for marker in markers)
}
runner_env.update({'CEG_WM_ROOT_KEY': root_key, 'HF_TOKEN': hf_token})
command = [
    sys.executable, '-m', 'experiments.run_geometry_v7_r0',
    '--repo-root', str(checkout), '--expected-exact', APPROVED_EXACT,
    '--result-dir', str(LOCAL_RESULT_DIR),
    '--syncseal-checkpoint', str(SYNCSEAL_CHECKPOINT),
]
try:
    completed = subprocess.run(
        command, cwd=checkout, env=runner_env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False,
    )
finally:
    root_key = hf_token = ''
    runner_env.clear()
print(completed.stdout.strip())
if completed.returncode not in (0, 2):
    raise RuntimeError('Geometry-V7 R0 runner stopped operationally')
if not (LOCAL_RESULT_DIR / 'result.json').is_file():
    raise RuntimeError('Geometry-V7 R0 complete result package is absent')

The subprocess invokes the reviewed real runner exactly once. Exit 0 and bounded exit 2 both require a complete local package; no unit is retried or removed. The runner reloads the fixed content-chain rosters, generates real step-18 U/C final RGB, applies official SyncSeal, recomputes paired Gate A/B from every final RGB, and records official eval_sync metrics.

In [ ]:
import shutil

if not LOCAL_RESULT_DIR.is_dir():
    raise RuntimeError('Run the bound real R0 producer before publication')
if DRIVE_RESULT_DIR.exists():
    raise FileExistsError(f'create-only Drive result already exists: {DRIVE_RESULT_DIR}')
DRIVE_RESULT_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_RESULT_DIR, DRIVE_RESULT_DIR)
print(DRIVE_RESULT_DIR)